# 02 — Iterative BOM Decomposition (Bottom-Up Arm)\n\nStarting from an R&S spectrum monitoring product, decompose level by level:\n\n**Product → Subsystem → Component → Sub-component → Fundamental Technology**\n\nEach level = one focused LLM call with only relevant RAG chunks. Leaf nodes get classified as Tech Drivers or not.

In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

from src.config import CHROMA_PERSIST_DIR, MAX_RAG_CHUNKS, BOM_MAX_DEPTH
from src.llm import embed, safe_chat_json
from src.traceability import BOMNode, TechDriver, DriverOrigin, DriverConfidence, KBPool
from src import prompts

import chromadb

client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)
collection = client.get_collection("knowledge_base")

# load KB state from notebook 01
with open("../data/outputs/kb_state.json") as f:
    kb_state = json.load(f)

print(f"KB: {len(kb_state['sources'])} sources, {len(kb_state['chunks'])} chunks")
print(f"ChromaDB: {collection.count()} documents")

KB: 18 sources, 30 chunks
ChromaDB: 30 documents


In [2]:
def retrieve_product(query: str, n: int = MAX_RAG_CHUNKS) -> list[dict]:
    """Retrieve from product pool only."""
    query_emb = embed([query])[0]
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=n,
        where={"pool": "product"},
        include=["documents", "metadatas"],
    )
    out = []
    for i in range(len(results["ids"][0])):
        out.append({
            "chunk_id": results["ids"][0][i],
            "content": results["documents"][0][i],
            "source_title": results["metadatas"][0][i]["source_title"],
        })
    return out


def format_rag_chunks(chunks: list[dict]) -> str:
    """Format chunks for prompt injection with chunk IDs for traceability."""
    parts = []
    for c in chunks:
        parts.append(f"[Chunk ID: {c['chunk_id']}] (Source: {c['source_title']})\n{c['content']}")
    return "\n\n---\n\n".join(parts)


# store all BOM nodes
bom_nodes: dict[str, BOMNode] = {}


def decompose(node: BOMNode, max_depth: int = BOM_MAX_DEPTH) -> BOMNode:
    """Recursively decompose a BOM node into sub-components."""
    if node.level >= max_depth:
        return node

    # RAG: retrieve chunks relevant to this specific component
    query = f"{node.name} {node.description} components sub-components technology"
    rag_chunks = retrieve_product(query, n=3)
    rag_text = format_rag_chunks(rag_chunks)

    # build parent context
    parent_context = "Top-level product" if node.parent_id is None else f"Part of: {bom_nodes[node.parent_id].name}"

    prompt = prompts.BOM_DECOMPOSE.format(
        parent_context=parent_context,
        component_name=node.name,
        component_description=node.description,
        rag_chunks=rag_text,
    )

    response = safe_chat_json(prompt, system="You are a technical analyst decomposing spectrum monitoring equipment into sub-components.")

    for comp in response.get("components", []):
        child = BOMNode(
            name=comp["name"],
            description=comp.get("description", ""),
            level=node.level + 1,
            parent_id=node.id,
            source_chunk_ids=response.get("source_chunk_ids_used", []),
        )
        bom_nodes[child.id] = child
        node.children_ids.append(child.id)

        indent = "  " * child.level
        leaf_marker = " 🔎" if comp.get("is_leaf") else ""
        print(f"{indent}├── {child.name}{leaf_marker}")

        if not comp.get("is_leaf", False):
            decompose(child, max_depth)

    return node

## Run Decomposition: R&S ESMW Ultra Wideband Monitoring Receiver

In [3]:
# start from the ESMW as our target product
root = BOMNode(
    name="R&S ESMW Ultra Wideband Monitoring Receiver",
    description="Next generation ultra wideband monitoring receiver for spectrum monitoring and direction finding. Frequency range 8 kHz to 40 GHz, up to 2 GHz real-time bandwidth, panorama scan at 2.6 THz/s, 75ns minimum signal duration for 100% POI, I/Q streaming via 100GE, AoA direction finding, TDOA localization, ITU-compliant.",
    level=0,
)
bom_nodes[root.id] = root

print(f"ROOT: {root.name}\n")
decompose(root, max_depth=4)
print(f"\nTotal BOM nodes: {len(bom_nodes)}")

ROOT: R&S ESMW Ultra Wideband Monitoring Receiver



  ├── Ultra Wideband RF Frontend


    ├── HF frequency range extension 🔎
    ├── 18 GHz frequency range extension 🔎
    ├── 40 GHz frequency range extension 🔎
    ├── Realtime bandwidth modules


      ├── 2 GHz realtime bandwidth module 🔎
      ├── 500 MHz realtime bandwidth module 🔎
      ├── Multi I/Q interface 🔎
      ├── 10 Gbit LAN interface 🔎
      ├── Digital downconverter 🔎
    ├── Multi I/Q interface 🔎
    ├── 10 Gbit LAN interface 🔎
    ├── Digital downconverter 🔎
  ├── High Sensitivity Receiver


    ├── RF Front-End


...

        ├── Rack-mountable 19-inch form factor enclosure 🔎
        ├── IP remote management interface 🔎
        ├── Environmental ruggedization and support 🔎
        ├── Low power consumption design 🔎
        ├── Compact form factor for concealed mounting 🔎
      ├── Networked Spectrum Monitoring Systems


        ├── Distributed RF Sensors 🔎
        ├── Centralized Geolocation Server
        ├── Network Architecture and IP Networking
        ├── Signal Processing and Software-Defined Approach
        ├── Spectrum Management Software Integration
    ├── Wide Frequency Coverage 🔎
    ├── High Sensitivity RF Reception 🔎
    ├── Wide Instantaneous Analysis Bandwidth 🔎
    ├── Data Processing and Digitization 🔎
    ├── Networked Monitoring Stations


      ├── Distributed RF Sensors 🔎
      ├── Centralized Geolocation Server


        ├── TDOA Geolocation Engine 🔎
        ├── RSS Geolocation Engine 🔎
        ├── Sensor Data Aggregation Module 🔎
        ├── Spectrum Management Integration Interface 🔎
        ├── Network Communication Module 🔎
        ├── Software-Defined Signal Processing Framework 🔎
      ├── IP Network Infrastructure 🔎
      ├── Geolocation Processing Algorithms 🔎
      ├── Spectrum Management Software Integration


        ├── Real-time Spectrum Visualization 🔎
        ├── 3D Geographic Display 🔎
        ├── Automated Signal Detection and Classification 🔎
        ├── Historical Data Analysis 🔎

Total BOM nodes: 130


## Classify Leaf Nodes as Tech Drivers

In [4]:
def get_bom_path(node_id: str) -> str:
    """Trace path from root to this node."""
    path = []
    current = bom_nodes[node_id]
    while current:
        path.append(current.name)
        current = bom_nodes.get(current.parent_id) if current.parent_id else None
    return " → ".join(reversed(path))


# find leaf nodes (no children)
leaves = [n for n in bom_nodes.values() if not n.children_ids]
print(f"Found {len(leaves)} leaf nodes to classify\n")

bom_drivers: list[TechDriver] = []

for leaf in leaves:
    bom_path = get_bom_path(leaf.id)
    prompt = prompts.BOM_CLASSIFY_DRIVER.format(
        name=leaf.name,
        description=leaf.description,
        bom_path=bom_path,
    )
    result = safe_chat_json(prompt, system="You are a technology analyst evaluating whether components represent active technology drivers.")

    leaf.is_tech_driver = result.get("is_tech_driver", False)
    marker = "✓ DRIVER" if leaf.is_tech_driver else "✗ passive"
    print(f"  [{marker}] {leaf.name} — {result.get('reasoning', '')[:80]}")

    if leaf.is_tech_driver:
        driver = TechDriver(
            name=leaf.name,
            description=f"{leaf.description}. {result.get('reasoning', '')}",
            origin=DriverOrigin.BOM,
            confidence=DriverConfidence.MEDIUM,
            bom_node_id=leaf.id,
            source_chunk_ids=leaf.source_chunk_ids,
        )
        bom_drivers.append(driver)

print(f"\n=== {len(bom_drivers)} Tech Drivers identified from BOM ===")

Found 98 leaf nodes to classify



  [✗ passive] HF frequency range extension — The HF frequency range extension module enables reception starting from 8 kHz up


  [✗ passive] 18 GHz frequency range extension — The 18 GHz frequency range extension is an incremental enhancement to extend the


  [✓ DRIVER] 40 GHz frequency range extension — The 40 GHz frequency range extension represents an active technology driver beca


  [✓ DRIVER] 2 GHz realtime bandwidth module — Realtime signal capture modules with 2 GHz instantaneous bandwidth represent a c


...

  [✓ DRIVER] Geolocation Processing Algorithms — Geolocation processing algorithms, particularly those based on TDOA and RSS tech


  [✓ DRIVER] Real-time Spectrum Visualization — Real-time Spectrum Visualization with 3D geographic mapping represents an active


  [✗ passive] 3D Geographic Display — 3D geographic display technology for visualizing spectrum data and emitter locat


  [✓ DRIVER] Automated Signal Detection and Classification — Automated Signal Detection and Classification involves advanced algorithms, ofte


  [✗ passive] Historical Data Analysis — Historical Data Analysis in regulatory frequency monitoring primarily involves p

=== 64 Tech Drivers identified from BOM ===


In [5]:
# deduplicate drivers by name (merge source_chunk_ids)
seen: dict[str, TechDriver] = {}
for d in bom_drivers:
    key = d.name.lower().strip()
    if key in seen:
        for sid in d.source_chunk_ids:
            if sid not in seen[key].source_chunk_ids:
                seen[key].source_chunk_ids.append(sid)
    else:
        seen[key] = d

bom_drivers = list(seen.values())
print(f"After dedup: {len(bom_drivers)} unique BOM drivers\n")
for d in bom_drivers:
    print(f"  • {d.name} ({len(d.source_chunk_ids)} sources)")

After dedup: 50 unique BOM drivers

  • 40 GHz frequency range extension (1 sources)
  • 2 GHz realtime bandwidth module (1 sources)
  • 500 MHz realtime bandwidth module (1 sources)
  • Multi I/Q interface (2 sources)
  • Digital downconverter (3 sources)
  • Wide Frequency Coverage Module (2 sources)
  • High Sensitivity Receiver Chain (2 sources)
  • Instantaneous Analysis Bandwidth Module (2 sources)
  • Digitizer (2 sources)
  • RF Front End (2 sources)
  • Low Noise Amplifier (LNA) (2 sources)
  • Intermediate Frequency Sampler (2 sources)
  • Receiver (2 sources)
  • Signal Analysis Module (2 sources)
  • Software-Defined Data Streaming (2 sources)
  • Edge Computing Integration (2 sources)
  • Software-defined Networking Components (2 sources)
  • Wide Instantaneous Bandwidth (2 sources)
  • High Scan Rates (2 sources)
  • 100 Gigabit Ethernet Interface (1 sources)
  • Digitized I/Q Data Stream (1 sources)
  • Angle of Arrival (AoA) Processing (1 sources)
  • Time Difference of

## Visualize BOM Tree & Save State

In [6]:
def print_tree(node_id: str, prefix: str = ""):
    node = bom_nodes[node_id]
    tag = ""
    if node.is_tech_driver:
        tag = " ⚡ DRIVER"
    elif not node.children_ids:
        tag = " (leaf)"
    print(f"{prefix}{node.name}{tag}")
    for i, child_id in enumerate(node.children_ids):
        is_last = i == len(node.children_ids) - 1
        child_prefix = prefix + ("└── " if is_last else "├── ")
        next_prefix = prefix + ("    " if is_last else "│   ")
        child = bom_nodes[child_id]
        child_tag = ""
        if child.is_tech_driver:
            child_tag = " ⚡ DRIVER"
        elif not child.children_ids:
            child_tag = " (leaf)"
        print(f"{child_prefix}{child.name}{child_tag}")
        if child.children_ids:
            for j, grandchild_id in enumerate(child.children_ids):
                gc_is_last = j == len(child.children_ids) - 1
                gc_prefix = next_prefix + ("└── " if gc_is_last else "├── ")
                gc_next = next_prefix + ("    " if gc_is_last else "│   ")
                gc = bom_nodes[grandchild_id]
                gc_tag = " ⚡ DRIVER" if gc.is_tech_driver else (" (leaf)" if not gc.children_ids else "")
                print(f"{gc_prefix}{gc.name}{gc_tag}")
                if gc.children_ids:
                    for k, ggc_id in enumerate(gc.children_ids):
                        ggc_last = k == len(gc.children_ids) - 1
                        ggc_prefix = gc_next + ("└── " if ggc_last else "├── ")
                        ggc = bom_nodes[ggc_id]
                        ggc_tag = " ⚡ DRIVER" if ggc.is_tech_driver else " (leaf)"
                        print(f"{ggc_prefix}{ggc.name}{ggc_tag}")

print("=== BOM TREE ===\n")
print_tree(root.id)

=== BOM TREE ===

R&S ESMW Ultra Wideband Monitoring Receiver
├── Ultra Wideband RF Frontend
│   ├── HF frequency range extension (leaf)
│   ├── 18 GHz frequency range extension (leaf)
│   ├── 40 GHz frequency range extension ⚡ DRIVER
│   ├── Realtime bandwidth modules
│   │   ├── 2 GHz realtime bandwidth module ⚡ DRIVER
│   │   ├── 500 MHz realtime bandwidth module ⚡ DRIVER
│   │   ├── Multi I/Q interface ⚡ DRIVER
│   │   ├── 10 Gbit LAN interface (leaf)
│   │   └── Digital downconverter ⚡ DRIVER
│   ├── Multi I/Q interface ⚡ DRIVER
│   ├── 10 Gbit LAN interface (leaf)
│   └── Digital downconverter (leaf)
├── High Sensitivity Receiver
│   ├── RF Front-End
│   │   ├── Wide Frequency Coverage Module ⚡ DRIVER
│   │   ├── High Sensitivity Receiver Chain ⚡ DRIVER
│   │   ├── High Linearity Front-End (leaf)
│   │   └── Instantaneous Analysis Bandwidth Module ⚡ DRIVER
│   ├── Digitizer ⚡ DRIVER
│   └── Signal Processing Unit
│       ├── Receiver (leaf)
│       ├── Digitizer (leaf)
│       └─

In [7]:
# save state for downstream notebooks
bom_state = {
    "bom_nodes": {k: v.model_dump(mode="json") for k, v in bom_nodes.items()},
    "bom_drivers": [d.model_dump(mode="json") for d in bom_drivers],
    "root_id": root.id,
}
with open("../data/outputs/bom_state.json", "w") as f:
    json.dump(bom_state, f, indent=2)

print(f"Saved {len(bom_nodes)} BOM nodes, {len(bom_drivers)} tech drivers")
print(f"\nBOM Drivers:")
for d in bom_drivers:
    print(f"  [{d.confidence.value}] {d.name}")
    print(f"    Sources: {d.source_chunk_ids[:3]}")
    print()

Saved 130 BOM nodes, 50 tech drivers

BOM Drivers:
  [medium] 40 GHz frequency range extension
    Sources: ['c119f1502fcf']

  [medium] 2 GHz realtime bandwidth module
    Sources: ['c119f1502fcf']

  [medium] 500 MHz realtime bandwidth module
    Sources: ['c119f1502fcf']

  [medium] Multi I/Q interface
    Sources: ['c119f1502fcf', '7cdf2c658fcf']

  [medium] Digital downconverter
    Sources: ['c119f1502fcf', '3b4509d15837', '4ab165499716']

  [medium] Wide Frequency Coverage Module
    Sources: ['ef5076c5a88d', '4ab165499716']

  [medium] High Sensitivity Receiver Chain
    Sources: ['ef5076c5a88d', '4ab165499716']

  [medium] Instantaneous Analysis Bandwidth Module
    Sources: ['ef5076c5a88d', '4ab165499716']

  [medium] Digitizer
    Sources: ['ef5076c5a88d', '4ab165499716']

  [medium] RF Front End
    Sources: ['ef5076c5a88d', '4ab165499716']

  [medium] Low Noise Amplifier (LNA)
    Sources: ['ef5076c5a88d', '4ab165499716']

  [medium] Intermediate Frequency Sampler
    Sour